In [19]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [20]:
# Create an API clients
from anthropic import Anthropic

client = Anthropic()
# NOTE: this notebook pins to Sonnet 4.5 because it needs two features that
# newer models dropped:
#   - `temperature`: removed on the Claude 5 family (claude-sonnet-5,
#     claude-opus-4-8), which returns a 400.
#   - assistant prefill (seeding the reply with a trailing assistant message,
#     used below for the ```json stop-sequence trick): returns a 400 on the
#     whole 4.6+ family, INCLUDING Sonnet 4.6.
# Sonnet 4.5 is the newest model that still accepts BOTH.
model = "claude-sonnet-4-5"

In [21]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return next(block.text for block in message.content if block.type == "text")

In [22]:
messages = []

add_user_message(messages,
    "Generate a very short event bridge rule as json"
)
text = chat(messages)
text

'```json\n{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}\n```'

In [23]:
messages = []

add_user_message(messages,
    "Generate a very short event bridge rule as json"
)
add_assistant_message(messages, "```json")
text = chat(messages, stop_sequences=["```"])
text

'\n{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}\n'

In [24]:
import json

json.loads(text.strip())

{'source': ['aws.ec2'],
 'detail-type': ['EC2 Instance State-change Notification'],
 'detail': {'state': ['running']}}

## Why this notebook pins to an older model — and the modern equivalent

The cells above use two features that newer Claude models removed:

- **`temperature`** — removed on the Claude 5 family (`claude-sonnet-5`, `claude-opus-4-8`) and Fable 5; sending it returns a 400. Once **adaptive thinking** became the default, the model itself owns the explore/exploit tradeoff that people reached for `temperature` to control. The replacement lever is `output_config={"effort": ...}` (`low`→`max`), plus prompting for stylistic variety. (`temperature=0` never actually guaranteed deterministic output anyway — there's nondeterminism in the serving stack regardless.)
- **Assistant prefill** (seeding the reply with a trailing `assistant` message — the `` ```json `` trick used above with a `` ``` `` stop sequence) — returns a 400 on the entire **4.6+ family**, *including Sonnet 4.6*. Two reasons: (1) on a thinking model the response must start with a thinking block, which collides with "the assistant turn starts with *this* text"; and (2) prefill was a steering/jailbreak vector (putting compliant words in the model's mouth), so closing it removes a safety bypass.

Sonnet 4.5 is the newest model that still accepts **both**, which is why `model` is pinned there.

### The fix: structured outputs

The legitimate goal of the prefill + stop-sequence trick was **"give me clean JSON, no prose, no ```` ``` ```` fences."** That's now a first-class API feature — **structured outputs** (`output_config.format`) — which constrains the response to a JSON schema and validates it, instead of pre-typing `` ```json `` and hoping the model continues correctly. The cell below achieves the same result on a current model with no `temperature`, no prefill, and no stop sequence.

In [25]:
# Modern equivalent on a current model (no temperature, no prefill, no stop sequence).
# Structured outputs constrains the response to a JSON schema, so the first text
# block is guaranteed to be valid JSON matching the shape we asked for.
import json

event_bridge_schema = {
    "type": "object",
    "properties": {
        "Name": {"type": "string"},
        "EventPattern": {
            "type": "object",
            "properties": {
                "source": {"type": "array", "items": {"type": "string"}},
                "detail-type": {"type": "array", "items": {"type": "string"}},
            },
            "required": ["source", "detail-type"],
            "additionalProperties": False,
        },
        "State": {"type": "string", "enum": ["ENABLED", "DISABLED"]},
    },
    "required": ["Name", "EventPattern", "State"],
    "additionalProperties": False,
}

response = client.messages.create(
    model="claude-sonnet-5",  # temperature removed, prefill 400s — but we need neither
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "Generate a very short event bridge rule as json"}
    ],
    output_config={
        "format": {"type": "json_schema", "schema": event_bridge_schema}
    },
)

# No ```json fences to strip, no stop sequence — it's already clean JSON.
raw = next(block.text for block in response.content if block.type == "text")
print(json.dumps(json.loads(raw), indent=2))

{
  "Name": "S3ObjectCreatedRule",
  "EventPattern": {
    "source": [
      "aws.s3"
    ],
    "detail-type": [
      "Object Created"
    ]
  },
  "State": "ENABLED"
}
